In [1]:
import pandas as pd
import numpy as np
import torch.optim as optim
import torch.nn as nn
import torch

# Charger les données depuis le fichier CSV et le nettoyage des données
chemin_vers_le_fichier = 'train.csv'
donnees = pd.read_csv(chemin_vers_le_fichier, decimal=',', sep=";", na_values=["#VALEUR!"], index_col="time")
donnees.index = pd.to_datetime(donnees.index, format='%d/%m/%Y %H:%M')

donnees = donnees.sample(n=100236, random_state=12).copy()

# Calculer la taille des deux parties x1 et x2
taille_x1 = int(3 * len(donnees) / 4)
taille_x2 = len(donnees) - taille_x1

# Diviser le DataFrame en deux parties x1 et x2
x1 = donnees.iloc[:taille_x1]
x2 = donnees.iloc[taille_x1:]

donnees = x1
donnees_test = x2

# Récupérer la colonne "Net Power (MW)" dans un nouveau DataFrame
donnees_labels = donnees['Net Power (MW)']
predict_label = donnees_test['Net Power (MW)']

# Supprimer la colonne "Net Power (MW)" du DataFrame initial
columns_to_drop = ["amb pressure", "Network Frequency (Hz)", "Lower Heating Value (Wh/Nm3)","Net Power (MW)"]
donnees.drop(columns=columns_to_drop, axis=1, inplace=True)
donnees_test.drop(columns=columns_to_drop, axis=1, inplace=True)

# Convertir les données d'entraînement et les étiquettes en tenseurs
train_tensor = torch.tensor(donnees.values, dtype=torch.float32)
train_label_tensor = torch.tensor(donnees_labels.values, dtype=torch.float32)

# Convertir les données de prédiction en tenseur
predict_tensor = torch.tensor(donnees_test.values, dtype=torch.float32)

# Définir le modèle du régresseur simple (MLP)
class SimpleRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRegressor, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

input_size = donnees.shape[1]
hidden_size = 16
output_size = 1

model = SimpleRegressor(input_size, hidden_size, output_size)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.2)

# Mettre le modèle en mode d'entraînement
model.train()

# Initialisation de la liste des pertes
losses = []

# Boucle d'entraînement
num_epochs = 200000  # Augmenter le nombre d'époques pour de meilleures performances

for epoch in range(num_epochs):
    # Passer les données dans le modèle
    outputs = model(train_tensor)

    # Calculer la perte
    loss = criterion(outputs.squeeze(), train_label_tensor)

    # Réinitialiser les gradients, effectuer la rétropropagation et mettre à jour les poids
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("run : ", epoch)

    # Ajouter la perte à la liste
    losses.append(loss.item())

model_params = model.state_dict()

print(model_params, "\n")

plt.plot(range(1, num_epochs+1), losses, label='Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Fonction de perte en fonction du temps (nombre d\'époques)')
plt.legend()
plt.grid(True)
plt.show()

# Charger les données depuis le fichier CSV et le nettoyage de données
chemin_vers_le_fichier = 'test.csv'
test = pd.read_csv(chemin_vers_le_fichier, decimal=',', sep=";", na_values=["#VALEUR!"], index_col="time")

test.index = pd.to_datetime(test.index, format='%d/%m/%Y %H:%M')

test.drop(columns=columns_to_drop, axis=1, inplace=True)

# Convertir les données d'entraînement en tenseurs
train_tensor = torch.tensor(test.values, dtype=torch.float32)

# Passer les nouvelles données dans le modèle pour obtenir les prédictions
with torch.no_grad():
    predictions = model(train_tensor)

predictions = predictions.numpy().flatten()

df_predictions = pd.DataFrame({
    'time': test.index,
    'Net Power (MW)': predictions,
})

df_predictions.to_csv('predictions.csv', date_format='%d/%m/%Y %H:%M', index=False, sep=';')
df_predictions.head()



run :  0
run :  1
run :  2
run :  3
run :  4
run :  5
run :  6
run :  7
run :  8
run :  9
run :  10
run :  11
run :  12
run :  13
run :  14
run :  15
run :  16
run :  17
run :  18
run :  19
run :  20
run :  21
run :  22
run :  23
run :  24
run :  25
run :  26
run :  27
run :  28
run :  29
run :  30
run :  31
run :  32
run :  33
run :  34
run :  35
run :  36
run :  37
run :  38
run :  39
run :  40
run :  41
run :  42
run :  43
run :  44
run :  45
run :  46
run :  47
run :  48
run :  49
run :  50
run :  51
run :  52
run :  53
run :  54
run :  55
run :  56
run :  57
run :  58
run :  59
run :  60
run :  61
run :  62
run :  63
run :  64
run :  65
run :  66
run :  67
run :  68
run :  69
run :  70
run :  71
run :  72
run :  73
run :  74
run :  75
run :  76
run :  77
run :  78
run :  79
run :  80
run :  81
run :  82
run :  83
run :  84
run :  85
run :  86
run :  87
run :  88
run :  89
run :  90
run :  91
run :  92
run :  93
run :  94
run :  95
run :  96
run :  97
run :  98
run :  99
run :  100

KeyboardInterrupt: 